In [14]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

# Load the dataset (limiting to the first 10,000 rows if necessary)
df = pd.read_csv("data/song_lyrics_en.csv").head(10000)

# Use 'song_document' as features and 'popularity_bin' as the target
X = df['song_document']
y = df['popularity_bin']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# Text Encoding: Bag of Words
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# Text Encoding: TF-IDF
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Model Training: SVC for Bag of Words
svc_bow = SVC()
svc_bow.fit(X_train_bow, y_train)
y_pred_bow = svc_bow.predict(X_test_bow)

# Model Training: SVC for TF-IDF
svc_tfidf = SVC()
svc_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = svc_tfidf.predict(X_test_tfidf)

# Evaluation: Accuracy and F1-score for Bag of Words
accuracy_bow = accuracy_score(y_test, y_pred_bow)
f1_bow = f1_score(y_test, y_pred_bow, average='weighted')

# Evaluation: Accuracy and F1-score for TF-IDF
accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)
f1_tfidf = f1_score(y_test, y_pred_tfidf, average='weighted')

# Display the results
print(f"Bag of Words - Accuracy: {accuracy_bow:.4f}, F1-score: {f1_bow:.4f}")
print(f"TF-IDF - Accuracy: {accuracy_tfidf:.4f}, F1-score: {f1_tfidf:.4f}")


Bag of Words - Accuracy: 0.5875, F1-score: 0.5043
TF-IDF - Accuracy: 0.5825, F1-score: 0.4794


In [26]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from scipy.sparse import hstack

# Load your dataset (limiting to the first 10,000 rows)
df = pd.read_csv("data/song_lyrics_en.csv").head(10000)

# Load the artist and tag embeddings
artist_embeddings = np.load('data/artist_embeddings.npy')
tag_embeddings = np.load('data/tag_embeddings.npy')

# Ensure the number of embeddings matches the number of rows in the dataset
assert artist_embeddings.shape[0] == df.shape[0], "Mismatch in number of artist embeddings"
assert tag_embeddings.shape[0] == df.shape[0], "Mismatch in number of tag embeddings"

# Use 'song_document' as features and 'popularity_bin' as labels
X = df['song_document']
y = df['popularity_bin']

# Split the data into training and testing sets, including artist and tag embeddings
X_train, X_test, y_train, y_test, artist_train, artist_test, tag_train, tag_test = train_test_split(
    X, y, artist_embeddings, tag_embeddings, test_size=0.2, random_state=42
)

# Text Encoding: Bag of Words
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# Text Encoding: TF-IDF
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Concatenate artist and tag embeddings with text features (Bag of Words)
X_train_combined_bow = hstack([X_train_bow, artist_train, tag_train])
X_test_combined_bow = hstack([X_test_bow, artist_test, tag_test])

# Concatenate artist and tag embeddings with text features (TF-IDF)
X_train_combined_tfidf = hstack([X_train_tfidf, artist_train, tag_train])
X_test_combined_tfidf = hstack([X_test_tfidf, artist_test, tag_test])

# Model Training: SVC for Bag of Words + embeddings
svc_bow = SVC()
svc_bow.fit(X_train_combined_bow, y_train)
y_pred_bow = svc_bow.predict(X_test_combined_bow)

# Model Training: SVC for TF-IDF + embeddings
svc_tfidf = SVC()
svc_tfidf.fit(X_train_combined_tfidf, y_train)
y_pred_tfidf = svc_tfidf.predict(X_test_combined_tfidf)

# Evaluation: Accuracy and F1-score for Bag of Words + embeddings
accuracy_bow = accuracy_score(y_test, y_pred_bow)
f1_bow = f1_score(y_test, y_pred_bow, average='weighted')

# Evaluation: Accuracy and F1-score for TF-IDF + embeddings
accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)
f1_tfidf = f1_score(y_test, y_pred_tfidf, average='weighted')

# Display the results
print(f"Bag of Words + Embeddings - Accuracy: {accuracy_bow:.4f}, F1-score: {f1_bow:.4f}")
print(f"TF-IDF + Embeddings - Accuracy: {accuracy_tfidf:.4f}, F1-score: {f1_tfidf:.4f}")


AssertionError: Mismatch in number of artist embeddings

In [28]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from scipy.sparse import hstack

# Load your dataset (limiting to the first 10,000 rows)
df = pd.read_csv("data/song_lyrics_en.csv").head(10000)

# Load the artist and tag embeddings
artist_embeddings = np.load('data/artist_embeddings.npy')
tag_embeddings = np.load('data/tag_embeddings.npy')

# Dynamically adjust dataset and embeddings to match the smallest size
min_size = min(df.shape[0], artist_embeddings.shape[0], tag_embeddings.shape[0])
df = df.head(min_size)
artist_embeddings = artist_embeddings[:min_size]
tag_embeddings = tag_embeddings[:min_size]

# Use 'song_document' as features and 'popularity_bin' as labels
X = df['song_document']
y = df['popularity_bin']

# Split the data into training and testing sets, including artist and tag embeddings
X_train, X_test, y_train, y_test, artist_train, artist_test, tag_train, tag_test = train_test_split(
    X, y, artist_embeddings, tag_embeddings, test_size=0.2, random_state=42
)

# Text Encoding: Bag of Words
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# Text Encoding: TF-IDF
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Concatenate artist and tag embeddings with text features (Bag of Words)
X_train_combined_bow = hstack([X_train_bow, artist_train, tag_train])
X_test_combined_bow = hstack([X_test_bow, artist_test, tag_test])

# Concatenate artist and tag embeddings with text features (TF-IDF)
X_train_combined_tfidf = hstack([X_train_tfidf, artist_train, tag_train])
X_test_combined_tfidf = hstack([X_test_tfidf, artist_test, tag_test])

# Model Training: SVC for Bag of Words + embeddings
svc_bow = SVC()
svc_bow.fit(X_train_combined_bow, y_train)
y_pred_bow = svc_bow.predict(X_test_combined_bow)

# Model Training: SVC for TF-IDF + embeddings
svc_tfidf = SVC()
svc_tfidf.fit(X_train_combined_tfidf, y_train)
y_pred_tfidf = svc_tfidf.predict(X_test_combined_tfidf)

# Evaluation: Accuracy and F1-score for Bag of Words + embeddings
accuracy_bow = accuracy_score(y_test, y_pred_bow)
f1_bow = f1_score(y_test, y_pred_bow, average='weighted')

# Evaluation: Accuracy and F1-score for TF-IDF + embeddings
accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)
f1_tfidf = f1_score(y_test, y_pred_tfidf, average='weighted')

# Display the results
print(f"Bag of Words + Embeddings - Accuracy: {accuracy_bow:.4f}, F1-score: {f1_bow:.4f}")
print(f"TF-IDF + Embeddings - Accuracy: {accuracy_tfidf:.4f}, F1-score: {f1_tfidf:.4f}")


TypeError: expected dimension <= 2 array or matrix

In [32]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from scipy.sparse import hstack

# Load your dataset (limiting to the first 10,000 rows)
df = pd.read_csv("data/song_lyrics_en.csv").head(10000)

# Load the artist and tag embeddings
artist_embeddings = np.load('data/artist_embeddings.npy')
tag_embeddings = np.load('data/tag_embeddings.npy')

# Dynamically adjust dataset and embeddings to match the smallest size
min_size = min(df.shape[0], artist_embeddings.shape[0], tag_embeddings.shape[0])
df = df.head(min_size)
artist_embeddings = artist_embeddings[:min_size]
tag_embeddings = tag_embeddings[:min_size]

# Reshape embeddings to ensure they are 2D
artist_embeddings = artist_embeddings.reshape(artist_embeddings.shape[0], -1)
tag_embeddings = tag_embeddings.reshape(tag_embeddings.shape[0], -1)

# Use 'song_document' as features and 'popularity_bin' as labels
X = df['song_document']
y = df['popularity_bin']

# Split the data into training and testing sets, including artist and tag embeddings
X_train, X_test, y_train, y_test, artist_train, artist_test, tag_train, tag_test = train_test_split(
    X, y, artist_embeddings, tag_embeddings, test_size=0.2, random_state=42
)

# Text Encoding: Bag of Words
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# Text Encoding: TF-IDF
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Concatenate artist and tag embeddings with text features (Bag of Words)
X_train_combined_bow = hstack([X_train_bow, artist_train, tag_train])
X_test_combined_bow = hstack([X_test_bow, artist_test, tag_test])

# Concatenate artist and tag embeddings with text features (TF-IDF)
X_train_combined_tfidf = hstack([X_train_tfidf, artist_train, tag_train])
X_test_combined_tfidf = hstack([X_test_tfidf, artist_test, tag_test])

# Model Training: SVC for Bag of Words + embeddings
svc_bow = SVC()
svc_bow.fit(X_train_combined_bow, y_train)
y_pred_bow = svc_bow.predict(X_test_combined_bow)

# Model Training: SVC for TF-IDF + embeddings
svc_tfidf = SVC()
svc_tfidf.fit(X_train_combined_tfidf, y_train)
y_pred_tfidf = svc_tfidf.predict(X_test_combined_tfidf)

# Evaluation: Accuracy and F1-score for Bag of Words + embeddings
accuracy_bow = accuracy_score(y_test, y_pred_bow)
f1_bow = f1_score(y_test, y_pred_bow, average='weighted')

# Evaluation: Accuracy and F1-score for TF-IDF + embeddings
accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)
f1_tfidf = f1_score(y_test, y_pred_tfidf, average='weighted')

# Display the results
print(f"Bag of Words + Embeddings - Accuracy: {accuracy_bow:.4f}, F1-score: {f1_bow:.4f}")
print(f"TF-IDF + Embeddings - Accuracy: {accuracy_tfidf:.4f}, F1-score: {f1_tfidf:.4f}")


Bag of Words + Embeddings - Accuracy: 0.5875, F1-score: 0.5043
TF-IDF + Embeddings - Accuracy: 0.6500, F1-score: 0.6143
